# 第 2 步：基础模型训练

本 notebook 完成以下任务：
1. 加载预处理好的数据
2. 训练 LSTM 基线模型（含评估、可视化、保存）
3. 训练 Transformer 基线模型（含评估、可视化、保存）
4. 模型对比与预测可视化

**使用的数据集**：ETTh1（小时级，少变量）、ETTm1（15分钟级，高频）、ECL（小时级，多变量321个）

## 1. 环境检查与导入

In [ ]:
import sys
import os
import json
import time
from datetime import datetime
from dataclasses import dataclass, fields

# 添加项目根目录到路径
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(ROOT)

import torch
import numpy as np
import matplotlib.pyplot as plt
from models import LSTMModel, TransformerModel, TimeSeriesDataset, Trainer
from torch.utils.data import DataLoader

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'设备: {"cuda" if torch.cuda.is_available() else "cpu"}')


def to_jsonable(value):
    """递归转换 NumPy 类型为 JSON 可序列化的 Python 类型。"""
    if isinstance(value, dict):
        return {k: to_jsonable(v) for k, v in value.items()}
    if isinstance(value, list):
        return [to_jsonable(v) for v in value]
    if isinstance(value, np.generic):
        return value.item()
    return value


def save_result(result, model_name, dataset, horizon, root):
    """将实验结果保存为 npy（含 history）和 json（摘要）两种格式。

    目录结构: test_results/h{horizon}/{dataset}/{model_name}/
    文件名带时间戳，避免覆盖历史结果。
    """
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    run_name = f"{dataset}_h{horizon}_{model_name}_{timestamp}"
    output_dir = os.path.join(root, 'test_results', f'h{horizon}', dataset, model_name)
    os.makedirs(output_dir, exist_ok=True)

    result_path = os.path.join(output_dir, f'{run_name}_results.npy')
    np.save(result_path, result, allow_pickle=True)

    summary = {k: v for k, v in result.items() if k != 'history'}
    summary_path = os.path.join(output_dir, f'{run_name}_summary.json')
    with open(summary_path, 'w', encoding='utf-8') as f:
        json.dump(to_jsonable(summary), f, ensure_ascii=False, indent=2)

    print(f'结果已保存至: {output_dir}/')
    print(f'  npy: {run_name}_results.npy')
    print(f'  json: {run_name}_summary.json')


def config_to_dict(cfg):
    """将 dataclass 配置转为字典，方便序列化。"""
    return {f.name: getattr(cfg, f.name) for f in fields(cfg)}

## 2. 数据集配置

## 评估指标说明

本项目使用以下 4 个指标评估模型预测质量（均基于**归一化后**的数据计算）：

| 指标 | 公式 | 含义 | 取值范围 |
|------|------|------|----------|
| **MSE**（均方误差） | $\frac{1}{n}\sum(y-\hat{y})^2$ | 对**大误差**敏感，是训练时的损失函数 | ≥ 0，越小越好 |
| **MAE**（平均绝对误差） | $\frac{1}{n}\sum|y-\hat{y}|$ | 直观反映**平均偏差大小**，对异常值不敏感 | ≥ 0，越小越好 |
| **MAPE**（平均绝对百分比误差） | $\frac{100\%}{n}\sum|\frac{y-\hat{y}}{y}|$ | 以**百分比**表示误差，便于跨数据集比较 | ≥ 0%，越小越好 |
| **R²**（决定系数） | $1 - \frac{\sum(y-\hat{y})^2}{\sum(y-\bar{y})^2}$ | 衡量模型**解释数据方差**的能力 | ≤ 1，越接近 1 越好 |

> **简单理解**：MSE/MAE 越小 = 预测越准；R² 越接近 1 = 模型拟合越好。

In [ ]:
# 数据集配置
DATASET = "ETTh1"            # 可选: ETTh1, ETTm1, ECL
HORIZON = 24                  # 预测步长: 24, 48, 96, 168, 336
BATCH_SIZE = 32
DATA_DIR = os.path.join(ROOT, 'data', 'processed')

print(f'数据集: {DATASET}')
print(f'预测步长: {HORIZON}')
print(f'批次大小: {BATCH_SIZE}')

## 2.1 模型配置

In [ ]:
@dataclass
class LSTMConfig:
    """LSTM 模型与训练超参数配置。"""
    hidden_size: int = 128
    num_layers: int = 2
    dropout: float = 0.1
    lr: float = 3e-4
    weight_decay: float = 0.0
    epochs: int = 100
    patience: int = 15
    seed: int = 216


@dataclass
class TransformerConfig:
    """Transformer 模型与训练超参数配置。"""
    d_model: int = 128
    nhead: int = 8
    num_layers: int = 2
    dim_feedforward: int = 256
    dropout: float = 0.1
    lr: float = 5e-5
    weight_decay: float = 0.0
    epochs: int = 100
    patience: int = 15
    seed: int = 216


lstm_cfg = LSTMConfig()
tf_cfg = TransformerConfig()

print('=== LSTM 配置 ===')
for f in fields(lstm_cfg):
    print(f'  {f.name}: {getattr(lstm_cfg, f.name)}')
print()
print('=== Transformer 配置 ===')
for f in fields(tf_cfg):
    print(f'  {f.name}: {getattr(tf_cfg, f.name)}')

## 3. 加载数据

In [ ]:
# 创建数据加载器
train_dataset = TimeSeriesDataset(DATA_DIR, DATASET, HORIZON, 'train')
val_dataset = TimeSeriesDataset(DATA_DIR, DATASET, HORIZON, 'val')
test_dataset = TimeSeriesDataset(DATA_DIR, DATASET, HORIZON, 'test')

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

input_size = train_dataset.input_size
target_idx = train_dataset.target_idx

print(f'输入特征数: {input_size}')
print(f'目标列索引: {target_idx}')
print(f'训练样本: {len(train_dataset)}')
print(f'验证样本: {len(val_dataset)}')
print(f'测试样本: {len(test_dataset)}')

## 4. 训练 LSTM 模型

In [ ]:
# 创建 LSTM 模型
lstm_model = LSTMModel(
    input_size=input_size,
    hidden_size=lstm_cfg.hidden_size,
    num_layers=lstm_cfg.num_layers,
    dropout=lstm_cfg.dropout,
    horizon=HORIZON
)

print(f'LSTM 模型参数量: {sum(p.numel() for p in lstm_model.parameters()):,}')
print(f'配置: hidden_size={lstm_cfg.hidden_size}, num_layers={lstm_cfg.num_layers}, dropout={lstm_cfg.dropout}')

In [ ]:
# 训练 LSTM（使用 GPU，固定随机种子）
lstm_trainer = Trainer(lstm_model, device='cuda', lr=lstm_cfg.lr, seed=lstm_cfg.seed)

lstm_start = time.time()
lstm_history = lstm_trainer.train(
    train_loader, val_loader,
    epochs=lstm_cfg.epochs,
    patience=lstm_cfg.patience,
    save_dir=os.path.join(ROOT, 'checkpoints'),
    model_name=f'LSTM_{DATASET}_h{HORIZON}',
    log_dir=os.path.join(ROOT, 'runs/LSTM')
)
lstm_train_time = time.time() - lstm_start

In [ ]:
# 检查 history 包含的键
print("lstm_history 的键:", list(lstm_history.keys()))

In [ ]:
# 绘制 LSTM 训练曲线
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 损失曲线
axes[0].plot(lstm_history['train_losses'], label='训练损失')
axes[0].plot(lstm_history['val_losses'], label='验证损失')
axes[0].set_title('LSTM 损失曲线')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].legend()
axes[0].grid(True)

# R² 曲线（准确率）
axes[1].plot(lstm_history['train_r2'], label='训练 R²')
axes[1].plot(lstm_history['val_r2'], label='验证 R²')
axes[1].set_title('LSTM R² (准确率)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('R²')
axes[1].legend()
axes[1].grid(True)

# 测试 LSTM
lstm_preds, lstm_targets = lstm_trainer.predict(test_loader)
lstm_metrics = lstm_trainer.compute_metrics(lstm_preds, lstm_targets, target_idx)

# 测试指标
metrics_names = ['MSE', 'MAE', 'MAPE', 'R2']
values = [lstm_metrics['MSE'], lstm_metrics['MAE'], lstm_metrics['MAPE'], lstm_metrics['R2']]
axes[2].bar(metrics_names, values, color=['blue', 'orange', 'green', 'red'])
axes[2].set_title('LSTM 测试指标')
axes[2].grid(True)

plt.tight_layout()
plt.show()

print(f'LSTM 测试结果:')
print(f'  MSE: {lstm_metrics["MSE"]:.6f}')
print(f'  MAE: {lstm_metrics["MAE"]:.6f}')
print(f'  MAPE: {lstm_metrics["MAPE"]:.2f}%')
print(f'  R² (准确率): {lstm_metrics["R2"]:.4f}')
print(f'  MSE (目标列): {lstm_metrics["MSE_target"]:.6f}')
print(f'  R² (目标列): {lstm_metrics["R2_target"]:.4f}')

In [ ]:
# ===================== 保存 LSTM 结果 =====================
lstm_result = {
    "dataset": DATASET,
    "horizon": HORIZON,
    "model": "lstm",
    "epochs": lstm_cfg.epochs,
    "trained_epochs": len(lstm_history["train_losses"]),
    "best_epoch": int(np.argmin(lstm_history["val_losses"]) + 1),
    "patience": lstm_cfg.patience,
    "batch_size": BATCH_SIZE,
    "learning_rate": lstm_cfg.lr,
    "weight_decay": lstm_cfg.weight_decay,
    "device": lstm_trainer.device,
    "seed": lstm_cfg.seed,
    "run_tag": "notebook",
    "data_dir": DATA_DIR,
    "sample_limit": 0,
    "train_samples": len(train_dataset),
    "val_samples": len(val_dataset),
    "test_samples": len(test_dataset),
    "input_size": input_size,
    "target_idx": target_idx,
    "model_params": sum(p.numel() for p in lstm_model.parameters()),
    "train_time_seconds": lstm_train_time,
    "history": lstm_history,
    "metrics": lstm_metrics,
    "best_val_loss": lstm_history["best_val_loss"],
    "best_val_r2": lstm_history["best_val_r2"],
    "config": config_to_dict(lstm_cfg),
}
save_result(lstm_result, "lstm", DATASET, HORIZON, ROOT)

## 5. 训练 Transformer 模型

In [ ]:
# 创建 Transformer 模型
transformer_model = TransformerModel(
    input_size=input_size,
    d_model=tf_cfg.d_model,
    nhead=tf_cfg.nhead,
    num_layers=tf_cfg.num_layers,
    dim_feedforward=tf_cfg.dim_feedforward,
    dropout=tf_cfg.dropout,
    horizon=HORIZON
)

print(f'Transformer 模型参数量: {sum(p.numel() for p in transformer_model.parameters()):,}')
print(f'配置: d_model={tf_cfg.d_model}, nhead={tf_cfg.nhead}, num_layers={tf_cfg.num_layers}, dim_ff={tf_cfg.dim_feedforward}')

In [ ]:
# 训练 Transformer（使用 GPU，固定随机种子）
transformer_trainer = Trainer(transformer_model, device='cuda', lr=tf_cfg.lr, seed=tf_cfg.seed)

tf_start = time.time()
transformer_history = transformer_trainer.train(
    train_loader, val_loader,
    epochs=tf_cfg.epochs,
    patience=tf_cfg.patience,
    save_dir=os.path.join(ROOT, 'checkpoints'),
    model_name=f'Transformer_{DATASET}_h{HORIZON}',
    log_dir=os.path.join(ROOT, 'runs/Transformer')
)
tf_train_time = time.time() - tf_start

In [ ]:
# 绘制 Transformer 训练曲线
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 损失曲线
axes[0].plot(transformer_history['train_losses'], label='训练损失')
axes[0].plot(transformer_history['val_losses'], label='验证损失')
axes[0].set_title('Transformer 损失曲线')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].legend()
axes[0].grid(True)

# R² 曲线（准确率）
axes[1].plot(transformer_history['train_r2'], label='训练 R²')
axes[1].plot(transformer_history['val_r2'], label='验证 R²')
axes[1].set_title('Transformer R² (准确率)')
axes[1].set_xlabel('Epoch')
axes[2].set_ylabel('R²')
axes[1].legend()
axes[1].grid(True)

# 测试 Transformer
transformer_preds, transformer_targets = transformer_trainer.predict(test_loader)
transformer_metrics = transformer_trainer.compute_metrics(transformer_preds, transformer_targets, target_idx)

# 测试指标
metrics_names = ['MSE', 'MAE', 'MAPE', 'R2']
values = [transformer_metrics['MSE'], transformer_metrics['MAE'], transformer_metrics['MAPE'], transformer_metrics['R2']]
axes[2].bar(metrics_names, values, color=['blue', 'orange', 'green', 'red'])
axes[2].set_title('Transformer 测试指标')
axes[2].grid(True)

plt.tight_layout()
plt.show()

print(f'Transformer 测试结果:')
print(f'  MSE: {transformer_metrics["MSE"]:.6f}')
print(f'  MAE: {transformer_metrics["MAE"]:.6f}')
print(f'  MAPE: {transformer_metrics["MAPE"]:.2f}%')
print(f'  R² (准确率): {transformer_metrics["R2"]:.4f}')
print(f'  MSE (目标列): {transformer_metrics["MSE_target"]:.6f}')
print(f'  R² (目标列): {transformer_metrics["R2_target"]:.4f}')

In [ ]:
# ===================== 保存 Transformer 结果 =====================
transformer_result = {
    "dataset": DATASET,
    "horizon": HORIZON,
    "model": "transformer",
    "epochs": tf_cfg.epochs,
    "trained_epochs": len(transformer_history["train_losses"]),
    "best_epoch": int(np.argmin(transformer_history["val_losses"]) + 1),
    "patience": tf_cfg.patience,
    "batch_size": BATCH_SIZE,
    "learning_rate": tf_cfg.lr,
    "weight_decay": tf_cfg.weight_decay,
    "device": transformer_trainer.device,
    "seed": tf_cfg.seed,
    "run_tag": "notebook",
    "data_dir": DATA_DIR,
    "sample_limit": 0,
    "train_samples": len(train_dataset),
    "val_samples": len(val_dataset),
    "test_samples": len(test_dataset),
    "input_size": input_size,
    "target_idx": target_idx,
    "model_params": sum(p.numel() for p in transformer_model.parameters()),
    "train_time_seconds": tf_train_time,
    "history": transformer_history,
    "metrics": transformer_metrics,
    "best_val_loss": transformer_history["best_val_loss"],
    "best_val_r2": transformer_history["best_val_r2"],
    "config": config_to_dict(tf_cfg),
}
save_result(transformer_result, "transformer", DATASET, HORIZON, ROOT)

## 6. 模型对比

In [ ]:
# 模型对比
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 训练曲线对比
axes[0].plot(lstm_history['val_losses'], label='LSTM')
axes[0].plot(transformer_history['val_losses'], label='Transformer')
axes[0].set_title('验证损失对比')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].legend()
axes[0].grid(True)

# R² 对比（准确率）
axes[1].plot(lstm_history['val_r2'], label='LSTM')
axes[1].plot(transformer_history['val_r2'], label='Transformer')
axes[1].set_title('验证 R² (准确率) 对比')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('R²')
axes[1].legend()
axes[1].grid(True)

# 测试指标对比
metrics_names = ['MSE', 'MAE', 'MAPE', 'R2']
x = np.arange(len(metrics_names))
width = 0.35

axes[2].bar(x - width/2, [lstm_metrics[m] for m in metrics_names], width, label='LSTM')
axes[2].bar(x + width/2, [transformer_metrics[m] for m in metrics_names], width, label='Transformer')
axes[2].set_title('测试指标对比')
axes[2].set_xticks(x)
axes[2].set_xticklabels(metrics_names)
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()

# 打印对比表
print('\n模型对比:')
print('-' * 60)
print(f'{"指标":<10} {"LSTM":<15} {"Transformer":<15}')
print('-' * 60)
for m in metrics_names:
    print(f'{m:<10} {lstm_metrics[m]:<15.6f} {transformer_metrics[m]:<15.6f}')
print('-' * 60)

## 7. 预测可视化

In [ ]:
# 选择一个样本进行可视化
sample_idx = 0
n_steps = min(200, HORIZON)  # 显示前 200 个时间步

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# LSTM 预测
axes[0].plot(lstm_targets[sample_idx, :n_steps, target_idx], label='真实值', linewidth=2)
axes[0].plot(lstm_preds[sample_idx, :n_steps, target_idx], label='LSTM 预测', linestyle='--')
axes[0].set_title(f'LSTM 预测 vs 真实值 (样本 {sample_idx})')
axes[0].set_xlabel('时间步')
axes[0].set_ylabel('归一化值')
axes[0].legend()
axes[0].grid(True)

# Transformer 预测
axes[1].plot(transformer_targets[sample_idx, :n_steps, target_idx], label='真实值', linewidth=2)
axes[1].plot(transformer_preds[sample_idx, :n_steps, target_idx], label='Transformer 预测', linestyle='--')
axes[1].set_title(f'Transformer 预测 vs 真实值 (样本 {sample_idx})')
axes[1].set_xlabel('时间步')
axes[1].set_ylabel('归一化值')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# 打印最终结果
print('\n' + '=' * 60)
print('基础模型训练完成')
print('=' * 60)
print(f'数据集: {DATASET}')
print(f'预测步长: {HORIZON}')
print()
print(f'--- LSTM ({lstm_cfg}) ---')
print(f'  参数量: {lstm_result["model_params"]:,}')
print(f'  训练耗时: {lstm_train_time:.1f}s  |  实际训练 epoch: {lstm_result["trained_epochs"]}')
print(f'  最佳验证损失: {lstm_result["best_val_loss"]:.6f}')
print(f'  最佳验证 R²: {lstm_result["best_val_r2"]:.4f}')
print()
print(f'--- Transformer ({tf_cfg}) ---')
print(f'  参数量: {transformer_result["model_params"]:,}')
print(f'  训练耗时: {tf_train_time:.1f}s  |  实际训练 epoch: {transformer_result["trained_epochs"]}')
print(f'  最佳验证损失: {transformer_result["best_val_loss"]:.6f}')
print(f'  最佳验证 R²: {transformer_result["best_val_r2"]:.4f}')
print()
print(f'模型已保存至: checkpoints/')
print(f'结果已保存至: test_results/h{HORIZON}/{DATASET}/')
print('=' * 60)

## 8. 总结